In [15]:
import pandas as pd
import sqlite3

In [16]:
df = pd.read_csv("../data/Telco_Churn_Cleaned.csv")

conn = sqlite3.connect("Customer_Churn.db")

df.to_sql(
    "customers",
    conn,
    if_exists="replace",
    index=False
)

print("Dataset successfully loaded into SQLite!")

Dataset successfully loaded into SQLite!


In [17]:
query = """
SELECT COUNT(*) AS total_customers
FROM customers;
"""

pd.read_sql_query(query, conn)

,total_customers
0,7043


#### What percentage of customers have churned?

In [18]:
query = """
SELECT
    Churn,
    COUNT(*) AS Total_Customers
FROM customers
GROUP BY Churn;
"""

pd.read_sql_query(query, conn)

,Churn,Total_Customers
0,No,5174
1,Yes,1869


In [19]:
query = """
SELECT
    Churn,
    COUNT(*) AS Total_Customers,
    ROUND(
        COUNT(*) * 100.0 /
        (SELECT COUNT(*) FROM customers),
        2
    ) AS Percentage
FROM customers
GROUP BY Churn;
"""

pd.read_sql_query(query, conn)

,Churn,Total_Customers,Percentage
0,No,5174,73.46
1,Yes,1869,26.54


#### Business Finding — Overall Customer Churn
Of the 7,043 customers analysed, 1,869 (26.54%) churned, while 5,174 (73.46%) remained. This provides a baseline for investigating which customer segments have higher churn rates.

#### Which contract types have the highest churn rates

In [20]:
query = """
SELECT
    Contract,
    COUNT(*) AS Churned_Customers
FROM customers
WHERE Churn = 'Yes'
GROUP BY Contract;
"""

pd.read_sql_query(query, conn)

,Contract,Churned_Customers
0,Month-to-month,1655
1,One year,166
2,Two year,48


In [21]:
query = """
SELECT
    Contract,
    COUNT(*) AS Total_Customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END)
        AS Churned_Customers,
    ROUND(
        100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS Churn_Rate
FROM customers
GROUP BY Contract
ORDER BY Churn_Rate DESC;
"""

pd.read_sql_query(query, conn)

,Contract,Total_Customers,Churned_Customers,Churn_Rate
0,Month-to-month,3875,1655,42.71
1,One year,1473,166,11.27
2,Two year,1695,48,2.83


Our analysis shows that month-to-month customers have a substantially higher churn rate (42.71%) than customers on one-year (11.27%) or two-year (2.83%) contracts.
This suggests that contract type is strongly associated with customer churn, although it doesn't establish that contract length causes customers to stay.

#### How does customer tenure affect churn?

In [22]:
query = """
SELECT
    tenure_group,
    COUNT(*) AS Total_Customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END)
        AS Churned_Customers,
    ROUND(
        100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS Churn_Rate
FROM customers
GROUP BY tenure_group
ORDER BY Churn_Rate DESC;
"""

pd.read_sql_query(query, conn)

,tenure_group,Total_Customers,Churned_Customers,Churn_Rate
0,0-6 months,1481,784,52.94
1,7-12 months,705,253,35.89
2,13-24 months,1024,294,28.71
3,25-48 months,1594,325,20.39
4,49-72 months,2239,213,9.51


Customers in their first six months have a churn rate of 52.94%, compared with just 9.51% among customers who have stayed for 49–72 months.
This suggests that early customer engagement and onboarding could be important areas for retention initiatives.

#### Which payment methods have the highest customer churn rates?

In [23]:
query = """
SELECT
    PaymentMethod,
    COUNT(*) AS Total_Customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END)
        AS Churned_Customers,
    ROUND(
        100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS Churn_Rate
FROM customers
GROUP BY PaymentMethod
ORDER BY Churn_Rate DESC;
"""

pd.read_sql_query(query, conn)

,PaymentMethod,Total_Customers,Churned_Customers,Churn_Rate
0,Electronic check,2365,1071,45.29
1,Mailed check,1612,308,19.11
2,Bank transfer (automatic),1544,258,16.71
3,Credit card (automatic),1522,232,15.24


Electronic check customers have the highest observed churn rate at 45.29%, considerably higher than customers using automatic payment methods.
This suggests an opportunity to investigate payment experiences and whether encouraging automatic payments could support retention. However, the data alone doesn't establish that changing payment methods would reduce churn.

#### Identifying high-risk customer segments

In [24]:
query = """
SELECT
    COUNT(*) AS Total_Customers
FROM customers
WHERE tenure <= 6
    AND Contract = 'Month-to-month';
"""

pd.read_sql_query(query, conn)

,Total_Customers
0,1413


In [25]:
query = """
SELECT
    COUNT(*) AS Total_Customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END)
        AS Churned_Customers,
    ROUND(
        100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS Churn_Rate
FROM customers
WHERE tenure <= 6
    AND Contract = 'Month-to-month';
"""

pd.read_sql_query(query, conn)

,Total_Customers,Churned_Customers,Churn_Rate
0,1413,780,55.2


In [26]:
query = """
SELECT
    COUNT(*) AS Total_Customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END)
        AS Churned_Customers,

    ROUND(
        100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS Churn_Rate,

    ROUND(
        100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END)
        / (
            SELECT COUNT(*)
            FROM customers
            WHERE Churn = 'Yes'
        ),
        2
    ) AS Churn_Contribution

FROM customers
WHERE tenure <= 6
    AND Contract = 'Month-to-month';
"""

pd.read_sql_query(query, conn)

,Total_Customers,Churned_Customers,Churn_Rate,Churn_Contribution
0,1413,780,55.2,41.73
